<a href="https://colab.research.google.com/github/tallapragadaananya-a11y/ML-2-1-Skill1/blob/main/Week%208/AdaBoost_Spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# ADABOOST vs XGBOOST - SPOTIFY DATASET
# ============================================================

import time
import numpy as np
import pandas as pd

from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount status:", e)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ============================================================
# SETTINGS
# ============================================================

RANDOM_STATE = 42

TARGET_COL = "Artist Type"

DATA_PATH = "/content/drive/MyDrive/Colab/Most Streamed Artists on Spotify.csv"


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

# Strip whitespaces from column names to prevent KeyErrors
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())


# ============================================================
# 2. REMOVE UNNECESSARY COLUMNS
# ============================================================

# Make a copy
df = df.copy()

# Remove rows where target is missing
df = df.dropna(subset=[TARGET_COL])


# ============================================================
# 3. CREATE TARGET
# ============================================================

print("\nArtist Type distribution:")
print(df[TARGET_COL].value_counts())


# Encode Artist Type into 0 and 1
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    df[TARGET_COL].astype(str)
)

print("\nTarget classes:")
for i, class_name in enumerate(target_encoder.classes_):
    print(i, "=", class_name)


# ============================================================
# 4. CREATE FEATURES
# ============================================================

X = df.drop(columns=[TARGET_COL])


# Identify categorical and numerical columns
cat_cols = X.select_dtypes(
    exclude="number"
).columns.tolist()

num_cols = X.select_dtypes(
    include="number"
).columns.tolist()


print("\nCategorical columns:")
print(cat_cols)

print("\nNumerical columns:")
print(num_cols)


# ============================================================
# 5. ENCODE CATEGORICAL VARIABLES
# ============================================================

encoders = {}

for c in cat_cols:

    le = LabelEncoder()

    X[c] = le.fit_transform(
        X[c].astype(str)
    )

    encoders[c] = le


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X[num_cols] = imputer.fit_transform(
    X[num_cols]
)


# ============================================================
# 7. SCALE NUMERICAL FEATURES
# ============================================================

scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(
    X[num_cols]
)


# ============================================================
# 8. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

VAL_SIZE = 0.15
TEST_SIZE = 0.15


# First create train + validation and test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)


# Calculate validation proportion
val_ratio = VAL_SIZE / (1 - TEST_SIZE)


# Split train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)


print(
    f"\nTrain: {X_train.shape} | "
    f"Val: {X_val.shape} | "
    f"Test: {X_test.shape}"
)


# ============================================================
# 9. BOOSTING BENCHMARK
# ============================================================

def boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
):

    results = []


    # ========================================================
    # MODEL 1: ADABOOST
    # ========================================================

    ada_base = DecisionTreeClassifier(
        max_depth=2,
        random_state=RANDOM_STATE
    )

    ada = AdaBoostClassifier(
        estimator=ada_base,
        n_estimators=200,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    )


    # Start timer
    t0 = time.time()

    ada.fit(
        X_train,
        y_train
    )

    ada_fit_time = time.time() - t0


    # Predictions
    ada_val_pred = ada.predict(
        X_val
    )

    ada_val_proba = ada.predict_proba(
        X_val
    )[:, 1]


    # Store results
    results.append({

        "model": "AdaBoost",

        "val_accuracy": accuracy_score(
            y_val,
            ada_val_pred
        ),

        "val_f1": f1_score(
            y_val,
            ada_val_pred
        ),

        "val_roc_auc": roc_auc_score(
            y_val,
            ada_val_proba
        ),

        "best_n_estimators": ada.n_estimators,

        "fit_time_sec": round(
            ada_fit_time,
            2
        )
    })


    # ========================================================
    # MODEL 2: XGBOOST
    # ========================================================

    xgb = XGBClassifier(

        n_estimators=1000,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        early_stopping_rounds=30,

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


    # Start timer
    t0 = time.time()

    xgb.fit(

        X_train,
        y_train,

        eval_set=[
            (X_val, y_val)
        ],

        verbose=False
    )

    xgb_fit_time = time.time() - t0


    # Predictions
    xgb_val_pred = xgb.predict(
        X_val
    )

    xgb_val_proba = xgb.predict_proba(
        X_val
    )[:, 1]


    # Store results
    results.append({

        "model": "XGBoost",

        "val_accuracy": accuracy_score(
            y_val,
            xgb_val_pred
        ),

        "val_f1": f1_score(
            y_val,
            xgb_val_pred
        ),

        "val_roc_auc": roc_auc_score(
            y_val,
            xgb_val_proba
        ),

        "best_n_estimators": xgb.best_iteration + 1,

        "fit_time_sec": round(
            xgb_fit_time,
            2
        )
    })


    # ========================================================
    # CREATE LEADERBOARD
    # ========================================================

    results_df = pd.DataFrame(
        results
    ).sort_values(
        "val_accuracy",
        ascending=False
    ).reset_index(
        drop=True
    )


    return results_df


# ============================================================
# 10. RUN BENCHMARK
# ============================================================

leaderboard = boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
)


print(
    "\nValidation leaderboard "
    "(sorted by val_accuracy):"
)

print(
    leaderboard.to_string(
        index=False
    )
)


# ============================================================
# 11. SAVE RESULTS
# ============================================================

leaderboard.to_csv(
    "spotify_boosting_benchmark_results.csv",
    index=False
)

print(
    "\nSaved results to "
    "spotify_boosting_benchmark_results.csv"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (500, 14)

Columns:
['Artist Name', 'Sex', 'Country of Origin', 'Primary Language', 'Primary Genre', 'Artist Type', 'Debut Year', 'Total Streams (in millions)', 'Lead Streams (in millions)', 'Feature Streams (in millions)', 'Solo Streams (in millions)', '% of Solo Streams', 'Collaborative Streams (in millions)', '% of Collaborative Streams']

First 5 rows:
     Artist Name     Sex Country of Origin Primary Language Primary Genre  \
0          Drake    Male            Canada          English       Hip-Hop   
1   Taylor Swift  Female     United States          English           Pop   
2      Bad Bunny    Male     United States          Spanish     Reggaeton   
3     The Weeknd    Male            Canada          English           R&B   
4  Justin Bieber    Male            Canada          English           Pop   

  Artist Type  Debut Year  Total S